# Velocity Model Building vs Seismic Unix
Runs every `velocityModel` step and its Seismic Unix equivalent on the same picks and compares the outputs.

Requires Seismic Unix (`$CWPROOT/bin` in `PATH`). SU computes in 32-bit floats and pyseiskit in 64-bit floats, so differences around 1e-6 of the velocities are rounding, not method differences.

In [ ]:
%pip install bokeh segyio

# *** Install pyseiskit from PyPI
# %pip install pyseiskit
# *** Install pyseiskit for local development
# %pip install -e ..
!uv pip install -e .. --reinstall

## 1. Data Setup

In [ ]:
import segyio
import numpy as np
from velanPicksReader import readVelanPicks

# Local files only: keep them in the 'demos' folder, next to this notebook
datasetFilename = 'line-1-line-2-data001_crsRegGathers_cdps.su'
picksFilename = 'line-1-line-2-data001_crsRegGathers_cdps.picksvelan.dat'

velocityPicks = readVelanPicks(picksFilename)
pickedCdps = np.array(sorted(velocityPicks))
pickingInterval = int(pickedCdps[1] - pickedCdps[0])

# Model geometry from the dataset headers: one column per CDP, same time sampling as the data
with segyio.su.open(datasetFilename, "r", ignore_geometry=True, endian='little') as seismicDataset:
    datasetCdps = seismicDataset.attributes(segyio.TraceField.CDP)[:]
    numberSamples = len(seismicDataset.samples)
    intervalTimeSamples = seismicDataset.header[0][segyio.TraceField.TRACE_SAMPLE_INTERVAL] / 1e6

cdps = np.arange(datasetCdps.min(), datasetCdps.max() + 1)
timeSamples = np.arange(numberSamples) * intervalTimeSamples

print(f"{len(pickedCdps)} picked CDPs from {pickedCdps[0]} to {pickedCdps[-1]}, every {pickingInterval} CDPs.")
print(f"Model: {numberSamples} samples every {intervalTimeSamples} s, CDPs {cdps[0]} to {cdps[-1]}.")

In [ ]:
import time
from seismicUnix import isSeismicUnixAvailable, runSeismicUnix
from pyseiskit import velocityModel

if not isSeismicUnixAvailable():
    raise RuntimeError("Seismic Unix programs were not found: add $CWPROOT/bin to PATH.")

benchmarkResults = []

def formatValues(values):
    return ",".join(repr(float(value)) for value in values)

def compareWithSeismicUnix(name, runPyseiskit, runSeismicUnixEquivalent):
    """Runs both implementations, stores timings and differences, and returns both outputs."""
    startTime = time.perf_counter()
    pyseiskitResult = runPyseiskit()
    pyseiskitSeconds = time.perf_counter() - startTime

    startTime = time.perf_counter()
    seismicUnixResult = runSeismicUnixEquivalent()
    seismicUnixSeconds = time.perf_counter() - startTime

    absoluteDifferences = np.abs(pyseiskitResult - seismicUnixResult)
    benchmarkResults.append({
        'name': name,
        'maxDifference': absoluteDifferences.max(),
        'relativeDifference': absoluteDifferences.max() / np.abs(seismicUnixResult).max(),
        'pyseiskitSeconds': pyseiskitSeconds,
        'seismicUnixSeconds': seismicUnixSeconds,
    })
    return pyseiskitResult, seismicUnixResult

def printBenchmarkTable(results):
    print(f"{'step':<46} {'max diff (m/s)':>15} {'max diff / max v':>17} {'pyseiskit (s)':>14} {'SU (s)':>9}")
    for result in results:
        print(
            f"{result['name']:<46} {result['maxDifference']:>15.3e} {result['relativeDifference']:>17.3e}"
            f" {result['pyseiskitSeconds']:>14.4f} {result['seismicUnixSeconds']:>9.4f}"
        )

## 2. Interpolation Along Time (SU `unisam`)
One `unisam xin=<tnmo> yin=<vnmo>` run per picked CDP, for every method.

In [ ]:
for method in ['linear', 'mono', 'akima', 'spline']:
    compareWithSeismicUnix(
        f"unisam method={method}",
        lambda: np.stack(
            [
                velocityModel.interpolateVelocityFunction(
                    velocityPicks[cdp]['times'], velocityPicks[cdp]['velocity'], numberSamples, intervalTimeSamples, method
                )
                for cdp in pickedCdps
            ],
            axis=1
        ),
        lambda: runSeismicUnix(
            " && ".join(
                f"unisam xin={formatValues(velocityPicks[cdp]['times'])} yin={formatValues(velocityPicks[cdp]['velocity'])}"
                f" nout={numberSamples} dxout={intervalTimeSamples} method={method} >> output.bin"
                for cdp in pickedCdps
            ),
            numberSamples
        )
    )

pickedVelocityFunctions = np.stack(
    [
        velocityModel.interpolateVelocityFunction(
            velocityPicks[cdp]['times'], velocityPicks[cdp]['velocity'], numberSamples, intervalTimeSamples
        )
        for cdp in pickedCdps
    ],
    axis=1
)
printBenchmarkTable(benchmarkResults[-4:])

## 3. Interpolation Across CDPs (SU `unisam2`)
`unisam2` applies `method1=` across CDPs (its 2nd dimension), the opposite of its self-documentation, so the time axis (already uniform) uses `method2=linear` and is left unchanged.

Cubic methods are compared inside the picked CDP range: outside it, both extrapolate the end polynomials (the full-range difference is printed below the table).

In [ ]:
isInsidePicks = (cdps >= pickedCdps[0]) & (cdps <= pickedCdps[-1])

def runUnisam2(method):
    return runSeismicUnix(
        f"unisam2 < input.bin nx1={numberSamples} dx1={intervalTimeSamples} n1={numberSamples} d1={intervalTimeSamples} f1=0"
        f" x2={formatValues(pickedCdps)} n2={len(cdps)} f2={cdps[0]} d2=1 method1={method} method2=linear > output.bin",
        numberSamples,
        pickedVelocityFunctions
    )

for method in ['linear', 'mono', 'akima', 'spline']:
    pyseiskitModel, seismicUnixModel = compareWithSeismicUnix(
        f"unisam2 method1={method} (picked CDP range)",
        lambda: velocityModel.interpolateAcrossCdps(pickedVelocityFunctions, pickedCdps, cdps, method)[:, isInsidePicks],
        lambda: runUnisam2(method)[:, isInsidePicks]
    )

printBenchmarkTable(benchmarkResults[-4:])

for method in ['mono', 'akima', 'spline']:
    fullRangeDifference = np.abs(
        velocityModel.interpolateAcrossCdps(pickedVelocityFunctions, pickedCdps, cdps, method) - runUnisam2(method)
    ).max()
    print(f"{method} on the full CDP range (extrapolated): max diff {fullRangeDifference:.3e} m/s")

## 4. Damped Least Squares Smoothing (SU `smooth2`)
On the linear model of the whole CDP range (`unisam` + `unisam2`).

In [ ]:
rmsVelocityModel = velocityModel.buildVelocityModel(velocityPicks, cdps, numberSamples, intervalTimeSamples)

for timeSmoothing, cdpSmoothing in [(5, 10), (10, 20), (20, 50)]:
    compareWithSeismicUnix(
        f"smooth2 r1={timeSmoothing} r2={cdpSmoothing}",
        lambda: velocityModel.applyDampedLeastSquaresSmoothing(rmsVelocityModel, timeSmoothing, cdpSmoothing),
        lambda: runSeismicUnix(
            f"smooth2 < input.bin n1={numberSamples} n2={len(cdps)} r1={timeSmoothing} r2={cdpSmoothing} > output.bin",
            numberSamples,
            rmsVelocityModel
        )
    )

printBenchmarkTable(benchmarkResults[-3:])

## 5. Moving Average Smoothing (SU `suconv` + `sumix`)
No SU program computes a 2D box mean, so the reference is a chain of trace programs (see the comments in `movingAverageWithSeismicUnix`). Windows are converted to half box sizes with the same rule as `applyMovingAverageSmoothing`.

In [ ]:
def movingAverageWithSeismicUnix(model, halfTimeSamples, halfCdps):
    """
    SU box mean: suconv (time) + sumix (CDPs) box sums of the model, divided by the
    same box sums of an all-ones model (number of samples inside the truncated box).
    """
    numberSamples = model.shape[0]
    timeBox = ",".join(["1"] * (2 * halfTimeSamples + 1))
    cdpBox = ",".join(["1"] * (2 * halfCdps + 1))

    # *** suconv is causal and appends 2*halfTimeSamples samples: suwind recentres it.
    # *** sumix is a trailing mix that skips its first traces and divides by the number of weights:
    # *** null traces on both sides, dropping the first 2*halfCdps outputs and sugain undo that.
    boxSum = (
        "boxSum() { "
        f"suconv < $1 filter={timeBox} | suwind itmin={halfTimeSamples} itmax={halfTimeSamples + numberSamples - 1} "
        "| sushw key=delrt a=0 > timeSum.su; "
        f"sunull nt={numberSamples} ntr={halfCdps} dt={intervalTimeSamples} > nullTraces.su; "
        f"cat nullTraces.su timeSum.su nullTraces.su | sumix mix={cdpBox} > mixed.su; "
        f"suwind < mixed.su skip={2 * halfCdps} | sugain scale={2 * halfCdps + 1} > $2; "
        "}; "
    )
    command = (
        boxSum
        + f"suaddhead ns={numberSamples} < input.bin | sushw key=dt a={round(intervalTimeSamples * 1e6)} > model.su && "
        + "sugain scale=0 < model.su | sugain bias=1 > ones.su && "
        + "boxSum model.su sums.su && boxSum ones.su counts.su && "
        + "suop2 sums.su counts.su op=quo | sustrip head=/dev/null outpar=/dev/null > output.bin"
    )
    return runSeismicUnix(command, numberSamples, model)

In [ ]:
movingAverageCases = [
    ('picked CDPs', pickedVelocityFunctions, 0.2, 3 * pickingInterval, pickingInterval),
    ('picked CDPs', pickedVelocityFunctions, 0.5, 5 * pickingInterval, pickingInterval),
    ('whole model', rmsVelocityModel, 0.2, 51, 1),
    ('whole model', rmsVelocityModel, 0.4, 201, 1),
]

for modelName, model, timeWindow, cdpWindow, intervalCdps in movingAverageCases:
    halfTimeSamples = round(timeWindow / intervalTimeSamples) // 2
    halfCdps = round(cdpWindow / intervalCdps) // 2
    compareWithSeismicUnix(
        f"moving average {modelName} {timeWindow} s x {cdpWindow} CDPs",
        lambda: velocityModel.applyMovingAverageSmoothing(model, timeWindow, cdpWindow, intervalTimeSamples, intervalCdps),
        lambda: movingAverageWithSeismicUnix(model, halfTimeSamples, halfCdps)
    )

printBenchmarkTable(benchmarkResults[-4:])

## 6. Velocity Conversion (SU `velconv`)
On the `smooth2 r1=10 r2=20` model.

- Dix (`vrmst` to `vintt`) differences come from 32-bit rounding of the `t·v²` differences in SU.
- Time to depth (`vintt` to `vintz`): while inverting z(t), SU accumulates the time axis in 32-bit floats (`yxtoxy`, `xi1 += dt`), which lags the exact time by about 2.4e-5 s at 3 s. Below the model's last time it switches to the exact time, so the single depth sample of each CDP that crosses the last time gets a spike (tens of m/s). pyseiskit has no such step; the other depth samples are printed separately.

In [ ]:
smoothedRmsVelocityModel = velocityModel.applyDampedLeastSquaresSmoothing(rmsVelocityModel, 10, 20)
intervalDepthSamples = 10.0
numberDepthSamples = 800

intervalVelocityModel, _ = compareWithSeismicUnix(
    "velconv intype=vrmst outtype=vintt",
    lambda: velocityModel.convertRmsToIntervalVelocity(smoothedRmsVelocityModel, intervalTimeSamples),
    lambda: runSeismicUnix(
        f"velconv < input.bin intype=vrmst outtype=vintt nt={numberSamples} dt={intervalTimeSamples} > output.bin",
        numberSamples,
        smoothedRmsVelocityModel
    )
)

pyseiskitDepthModel, seismicUnixDepthModel = compareWithSeismicUnix(
    "velconv intype=vintt outtype=vintz",
    lambda: velocityModel.convertTimeToDepthVelocity(intervalVelocityModel, intervalTimeSamples, numberDepthSamples, intervalDepthSamples),
    lambda: runSeismicUnix(
        f"velconv < input.bin intype=vintt outtype=vintz nt={numberSamples} dt={intervalTimeSamples}"
        f" nz={numberDepthSamples} dz={intervalDepthSamples} > output.bin",
        numberDepthSamples,
        intervalVelocityModel
    )
)

printBenchmarkTable(benchmarkResults[-2:])

# *** depth sample of each CDP that crosses the model's last time
depthsAtLastTime = 0.5 * intervalTimeSamples * intervalVelocityModel[:-1].sum(axis=0)
depthSamples = np.arange(numberDepthSamples)[:, None] * intervalDepthSamples
# *** 0.01 m of tolerance covers SU's 32-bit rounding of the last depth
isCrossingSample = (
    (depthSamples <= depthsAtLastTime + 0.01)
    & (depthSamples + intervalDepthSamples > depthsAtLastTime - 0.01)
)

depthDifferences = np.abs(pyseiskitDepthModel - seismicUnixDepthModel)
print(f"vintz max diff at the last-time crossing samples: {depthDifferences[isCrossingSample].max():.3e} m/s")
print(f"vintz max diff at every other depth sample: {depthDifferences[~isCrossingSample].max():.3e} m/s")

## 7. Whole Chain
The SU tutorial chain from picks to interval velocity in depth, as one shell pipeline: `unisam` per picked CDP → `unisam2` → `smooth2` → `velconv intype=vrmst outtype=vintz`. Its largest difference is the last-time crossing spike described in section 6.

In [ ]:
def buildDepthModelWithPyseiskit():
    model = velocityModel.buildVelocityModel(velocityPicks, cdps, numberSamples, intervalTimeSamples)
    model = velocityModel.applyDampedLeastSquaresSmoothing(model, 10, 20)
    model = velocityModel.convertRmsToIntervalVelocity(model, intervalTimeSamples)
    return velocityModel.convertTimeToDepthVelocity(model, intervalTimeSamples, numberDepthSamples, intervalDepthSamples)

def buildDepthModelWithSeismicUnix():
    timeInterpolation = " && ".join(
        f"unisam xin={formatValues(velocityPicks[cdp]['times'])} yin={formatValues(velocityPicks[cdp]['velocity'])}"
        f" nout={numberSamples} dxout={intervalTimeSamples} >> picked.bin"
        for cdp in pickedCdps
    )
    return runSeismicUnix(
        f"{timeInterpolation} && "
        f"unisam2 < picked.bin nx1={numberSamples} dx1={intervalTimeSamples} n1={numberSamples} d1={intervalTimeSamples} f1=0"
        f" x2={formatValues(pickedCdps)} n2={len(cdps)} f2={cdps[0]} d2=1 "
        f"| smooth2 n1={numberSamples} n2={len(cdps)} r1=10 r2=20 > smoothed.bin && "
        # *** velconv seeks its input, so it reads a file instead of a pipe
        f"velconv < smoothed.bin intype=vrmst outtype=vintz nt={numberSamples} dt={intervalTimeSamples}"
        f" nz={numberDepthSamples} dz={intervalDepthSamples} > output.bin",
        numberDepthSamples
    )

pyseiskitDepthModel, seismicUnixDepthModel = compareWithSeismicUnix(
    "whole chain picks -> vint(z)",
    buildDepthModelWithPyseiskit,
    buildDepthModelWithSeismicUnix
)
printBenchmarkTable(benchmarkResults[-1:])

In [ ]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import row
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.palettes import RdBu11
from pyseiskit import PALETTES
output_notebook()

def plotDepthModel(model, title, palette, valueRange, colorBarTitle):
    depthSamples = np.arange(numberDepthSamples) * intervalDepthSamples
    plotFigure = figure(
        width=420, height=500, title=title,
        x_axis_label="CDP", y_axis_label="Depth (m)",
        x_range=(cdps[0], cdps[-1]), y_range=(depthSamples[-1], depthSamples[0])
    )
    colorMapper = LinearColorMapper(palette=palette, low=valueRange[0], high=valueRange[1])
    plotFigure.image(
        image=[model], x=cdps[0], y=depthSamples[0],
        dw=cdps[-1] - cdps[0], dh=depthSamples[-1] - depthSamples[0], color_mapper=colorMapper
    )
    plotFigure.add_layout(ColorBar(color_mapper=colorMapper, title=colorBarTitle), 'right')
    return plotFigure

velocityRange = (seismicUnixDepthModel.min(), seismicUnixDepthModel.max())
depthModelDifference = pyseiskitDepthModel - seismicUnixDepthModel
differenceLimit = np.abs(depthModelDifference).max()
show(
    row(
        plotDepthModel(pyseiskitDepthModel, "pyseiskit", PALETTES['jet'], velocityRange, "m/s"),
        plotDepthModel(seismicUnixDepthModel, "Seismic Unix", PALETTES['jet'], velocityRange, "m/s"),
        plotDepthModel(depthModelDifference, "pyseiskit - Seismic Unix", RdBu11, (-differenceLimit, differenceLimit), "m/s")
    )
)

## 8. Summary

In [ ]:
printBenchmarkTable(benchmarkResults)